In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import os
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dense, Flatten, Dropout
from tensorflow.keras.optimizers import Adam
import matplotlib.pyplot as plt

In [ ]:
DATASET_PATH = "/content/drive/MyDrive/metal_defect_dataset/NEU Metal Surface Defects Data"

train_dir = os.path.join(DATASET_PATH, "train")
valid_dir = os.path.join(DATASET_PATH, "valid")
test_dir  = os.path.join(DATASET_PATH, "test")

print("Train:", train_dir)
print("Valid:", valid_dir)
print("Test:", test_dir)

Train: /content/drive/MyDrive/metal_defect_dataset/NEU Metal Surface Defects Data/train
Valid: /content/drive/MyDrive/metal_defect_dataset/NEU Metal Surface Defects Data/valid
Test: /content/drive/MyDrive/metal_defect_dataset/NEU Metal Surface Defects Data/test


In [ ]:
img_h, img_w = 224, 224
batch_size = 16

train_datagen = ImageDataGenerator(rescale=1/255.0, rotation_range=10,zoom_range=0.2,shear_range=0.1,
                horizontal_flip=True,)
test_datagen = ImageDataGenerator(rescale=1/255.0)

In [ ]:
train_gen = train_datagen.flow_from_directory(train_dir,target_size=(img_h, img_w),
    class_mode="categorical",batch_size=batch_size,)

Found 1656 images belonging to 6 classes.


In [ ]:
valid_gen = test_datagen.flow_from_directory(valid_dir,target_size=(img_h, img_w),
    class_mode="categorical",batch_size=batch_size,)

Found 72 images belonging to 6 classes.


In [ ]:
test_gen = test_datagen.flow_from_directory(test_dir,target_size=(img_h, img_w),class_mode="categorical",
    batch_size=batch_size,shuffle=False)

Found 72 images belonging to 6 classes.


In [ ]:
num_classes = len(train_gen.class_indices)
print("Classes:", train_gen.class_indices)


Classes: {'Crazing': 0, 'Inclusion': 1, 'Patches': 2, 'Pitted': 3, 'Rolled': 4, 'Scratches': 5}


In [ ]:
model = Sequential([Conv2D(32, (3,3), activation='relu', input_shape=(224,224,3)),MaxPooling2D(),
    Conv2D(64, (3,3), activation='relu'), MaxPooling2D(),
    Conv2D(128, (3,3), activation='relu'),MaxPooling2D(),
    Flatten(), Dense(256, activation='relu'),Dropout(0.4),Dense(num_classes, activation='softmax')])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
model.compile(loss="categorical_crossentropy",optimizer=Adam(1e-3),metrics=["accuracy"])


In [ ]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 86528)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │    22,151,424 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 6)              │         1,542 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 22,246,214 (84.86 MB)

 Trainable params: 22,246,214 (84.86 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history = model.fit(train_gen,epochs=10,validation_data=valid_gen)

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10
104/104 ━━━━━━━━━━━━━━━━━━━━ 706s 7s/step - accuracy: 0.2586 - loss: 1.8561 - val_accuracy: 0.7500 - val_loss: 0.8510
Epoch 2/10
104/104 ━━━━━━━━━━━━━━━━━━━━ 221s 2s/step - accuracy: 0.5770 - loss: 1.0418 - val_accuracy: 0.4583 - val_loss: 1.4682
Epoch 3/10
104/104 ━━━━━━━━━━━━━━━━━━━━ 255s 2s/step - accuracy: 0.7365 - loss: 0.7414 - val_accuracy: 0.8750 - val_loss: 0.3053
Epoch 4/10
104/104 ━━━━━━━━━━━━━━━━━━━━ 216s 2s/step - accuracy: 0.7693 - loss: 0.5713 - val_accuracy: 0.9167 - val_loss: 0.3100
Epoch 5/10
104/104 ━━━━━━━━━━━━━━━━━━━━ 222s 2s/step - accuracy: 0.7765 - loss: 0.6219 - val_accuracy: 0.9028 - val_loss: 0.2546
Epoch 6/10
104/104 ━━━━━━━━━━━━━━━━━━━━ 216s 2s/step - accuracy: 0.8504 - loss: 0.4039 - val_accuracy: 0.6806 - val_loss: 1.0973
Epoch 7/10
104/104 ━━━━━━━━━━━━━━━━━━━━ 262s 2s/step - accuracy: 0.8252 - loss: 0.4629 - val_accuracy: 0.9028 - val_loss: 0.3048
Epoch 8/10
104/104 ━━━━━━━━━━━━━━━━━━━━ 214s 2s/step - accuracy: 0.8604 - loss: 0.3943 - val_accu

In [ ]:
loss, acc = model.evaluate(test_gen)
print("Test Accuracy:", acc)

5/5 ━━━━━━━━━━━━━━━━━━━━ 20s 5s/step - accuracy: 0.8643 - loss: 0.4273
Test Accuracy: 0.7361111044883728


In [ ]:
model.save("/content/neu_defect_model.h5")
print("Model saved")


Model saved
